# Scaling Karşılaştırması — TF-IDF

TF-IDF vektörleştirme üzerine 8 farklı ölçekleme yönteminin etkisi.
Her scaler × her sınıflandırıcı kombinasyonu çalıştırılır ve karşılaştırmalı sonuçlar üretilir.

### Kütüphaneler

In [1]:
import pandas as pd
import numpy as np
import time
import copy
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier, NearestCentroid
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
import xgboost as xgb


### Veri Yükleme ve Bölme

In [2]:
df = pd.read_csv('spam.csv', encoding='latin-1')
df.drop(['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], axis=1, inplace=True)
df.columns = ['type', 'sms']
df['type'] = df['type'].map({'ham': 0, 'spam': 1})

X, y = df['sms'], df['type']
print(f"Veri seti: {len(df)} satır | ham: {(y==0).sum()}, spam: {(y==1).sum()}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=111, stratify=y
)
print(f"Eğitim: {len(X_train)} | Test: {len(X_test)} | Train spam oranı: {y_train.mean()*100:.1f}%")


Veri seti: 5572 satır | ham: 4825, spam: 747
Eğitim: 3900 | Test: 1672 | Train spam oranı: 13.4%


### TF-IDF Vektörleştirme

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec  = vectorizer.transform(X_test)

# Sparse matrix -> dense (scaler'lar için gerekli)
X_train_vec = X_train_vec.toarray()
X_test_vec  = X_test_vec.toarray()
print(f"Dense shape: {X_train_vec.shape}")
print(f"TF-IDF özellik boyutu: {X_train_vec.shape[1]}")


Dense shape: (3900, 7167)
TF-IDF özellik boyutu: 7167


### Değerlendirme Fonksiyonu

In [4]:
import scipy.stats as stats

def evaluate(name, clf, X_tr, y_tr, X_te, y_te):
    t0 = time.time()
    clf.fit(X_tr, y_tr)
    elapsed = time.time() - t0
    y_pred = clf.predict(X_te)
    try:
        y_score = clf.predict_proba(X_te)[:, 1]
    except AttributeError:
        y_score = clf.decision_function(X_te)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(clf, X_tr, y_tr, cv=cv, scoring='accuracy')
    cv_acc = cv_scores.mean()
    cv_std = cv_scores.std()
    ci = stats.t.interval(0.95, df=len(cv_scores)-1, loc=cv_acc, scale=stats.sem(cv_scores))
    return {
        'Model':        name,
        'Accuracy':     round(accuracy_score(y_te, y_pred), 4),
        'Precision':    round(precision_score(y_te, y_pred, zero_division=0), 4),
        'Recall':       round(recall_score(y_te, y_pred), 4),
        'F1':           round(f1_score(y_te, y_pred), 4),
        'ROC-AUC':      round(roc_auc_score(y_te, y_score), 4),
        'CV-Acc':       round(cv_acc, 4),
        'CV-Std':       round(cv_std, 4),
        'CI-95% Low':   round(ci[0], 4),
        'CI-95% High':  round(ci[1], 4),
        'Time(s)':      round(elapsed, 4),
    }

### Tüm Scaler'lar × Tüm Sınıflandırıcılar

8 farklı ölçekleme yöntemi sırayla uygulanır ve sonuçlar tek tabloda karşılaştırılır:
StandardScaler, MinMaxScaler, RobustScaler, Normalizer, Binarizer, PowerTransformer, QuantileTransformer, MaxAbsScaler

In [5]:
from sklearn.preprocessing import (StandardScaler, MinMaxScaler, RobustScaler,
                                    Normalizer, Binarizer, PowerTransformer,
                                    QuantileTransformer, MaxAbsScaler)

scalers = {
    'StandardScaler':      StandardScaler(),
    'MinMaxScaler':        MinMaxScaler(),
    'RobustScaler':        RobustScaler(),
    'Normalizer':          Normalizer(norm='l2'),
    'Binarizer':           Binarizer(threshold=0.0),
    'PowerTransformer':    PowerTransformer(method='yeo-johnson'),
    'QuantileTransformer': QuantileTransformer(output_distribution='normal', random_state=42),
    'MaxAbsScaler':        MaxAbsScaler(),
}

classifiers = [
    ('LR',  LogisticRegression(max_iter=1000)),
    ('DT',  DecisionTreeClassifier(random_state=42)),
    ('KNN', KNeighborsClassifier(n_neighbors=3)),
    ('SVM', SVC(kernel='linear', probability=True)),
    ('NC',  NearestCentroid()),
]

all_rows = []
total = len(scalers) * len(classifiers)
done  = 0

for scaler_name, scaler in scalers.items():
    X_tr_s = scaler.fit_transform(X_train_vec)
    X_te_s = scaler.transform(X_test_vec)
    for clf_name, clf in classifiers:
        t0 = time.time()
        row = evaluate(clf_name, copy.deepcopy(clf), X_tr_s, y_train, X_te_s, y_test)
        row['Scaler'] = scaler_name
        all_rows.append(row)
        done += 1
        elapsed = time.time() - t0
        print(f"[{done:2d}/{total}] {scaler_name:22s} + {clf_name:3s}  "
              f"Acc={row['Accuracy']:.4f}  F1={row['F1']:.4f}  ({elapsed:.1f}s)")
    print()

cols = ['Scaler', 'Model', 'Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC',
        'CV-Acc', 'CV-Std', 'CI-95% Low', 'CI-95% High', 'Time(s)']
df_results = pd.DataFrame(all_rows)[cols]
print(df_results.to_string(index=False))

[ 1/40] StandardScaler         + LR   Acc=0.9755  F1=0.9002  (1.2s)
[ 2/40] StandardScaler         + DT   Acc=0.9665  F1=0.8685  (17.3s)
[ 3/40] StandardScaler         + KNN  Acc=0.8929  F1=0.3395  (0.8s)
[ 4/40] StandardScaler         + SVM  Acc=0.9749  F1=0.8990  (278.4s)
[ 5/40] StandardScaler         + NC   Acc=0.9743  F1=0.8938  (0.5s)

[ 6/40] MinMaxScaler           + LR   Acc=0.9827  F1=0.9311  (0.9s)
[ 7/40] MinMaxScaler           + DT   Acc=0.9665  F1=0.8685  (15.4s)
[ 8/40] MinMaxScaler           + KNN  Acc=0.9175  F1=0.5548  (0.8s)
[ 9/40] MinMaxScaler           + SVM  Acc=0.9827  F1=0.9327  (134.6s)
[10/40] MinMaxScaler           + NC   Acc=0.9701  F1=0.8821  (0.4s)

[11/40] RobustScaler           + LR   Acc=0.9635  F1=0.8448  (1.5s)
[12/40] RobustScaler           + DT   Acc=0.9665  F1=0.8685  (16.7s)
[13/40] RobustScaler           + KNN  Acc=0.9611  F1=0.8441  (0.7s)
[14/40] RobustScaler           + SVM  Acc=0.9844  F1=0.9398  (157.3s)
[15/40] RobustScaler           + NC  